# Ingestão SIH/SUS e agregação — Diabetes no SUS

Este notebook baixa os microdados do SIH/SUS (Sistema de Informações Hospitalares),
filtra as internações por diabetes (CID-10 E10-E14), aplica a padronização de idade
e faixa etária do projeto, agrega tudo até a camada `silver` com PySpark e, por fim,
junta com população e cobertura de Atenção Primária à Saúde (APS) para gerar a
camada `gold` (`municipio_ano.csv`), usada nas análises de desigualdade regional no
cuidado ao diabetes.

**Este notebook roda no Google Colab, não localmente.** Os microdados do DATASUS são
distribuídos em formato `.dbc` (DBF compactado), que exige a biblioteca
`datasus-dbc` — sem *wheel* disponível para Python 3.13 no Windows, o que já foi
verificado e está documentado na Seção 2.4 do spec do projeto. Por isso todo o
processamento de ingestão e a agregação Spark acontecem no ambiente do Colab
(Linux, Python compatível), e apenas o resultado final (`municipio_ano.csv`) volta
para a máquina local.

**Janela de dados:** 2019 a 2024, as 27 UFs, 1.944 arquivos mensais esperados
(27 UFs × 6 anos × 12 meses).

**Regras de recorte:**
- Diagnóstico principal (`DIAG_PRINC`) começando em E10, E11, E12, E13 ou E14.
- Amputação de membro inferior: procedimento SIGTAP com prefixo `040805`.
- AIHs de continuação (`IDENT == 5`) são excluídas — não são novas internações.

## Parte 1 — Camada bronze (ingestão)

As células a seguir baixam cada arquivo mensal do FTP do DATASUS, descompactam o
`.dbc`, filtram as internações de diabetes e gravam um parquet particionado por UF
e ano em `bronze/uf=<UF>/ano=<AAAA>/RD<UF><AAMM>.parquet` no Google Drive.

### 1.1 Configuração do ambiente

Monta o Google Drive (onde os dados intermediários e os módulos do projeto ficam
armazenados, já que o Colab não tem disco persistente entre sessões) e instala as
bibliotecas necessárias:

- `datasus-dbc`: descompacta o formato `.dbc` do DATASUS para `.dbf`.
- `dbfread`: lê o `.dbf` resultante como um DataFrame do pandas.
- `pyarrow`: grava os arquivos intermediários em parquet.

**Correção aplicada:** o brief original desta etapa não incluía `dbfread` no
`pip install`, mas a célula de ingestão (Seção 1.4) faz `from dbfread import DBF` —
sem essa dependência instalada a ingestão falharia logo no primeiro arquivo.

In [ ]:
!pip install -q datasus-dbc pyarrow dbfread
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/diabetes_sus'
os.makedirs(f'{BASE}/bronze', exist_ok=True)
os.makedirs(f'{BASE}/logs', exist_ok=True)
print(BASE)

### 1.2 Módulos do projeto

As funções de filtro, conversão de idade e compatibilização de códigos de
município usadas neste notebook vêm do pacote `src/diabetes_sus/` do repositório
(o mesmo pacote usado e testado nas tarefas anteriores) — elas não são
reimplementadas aqui para evitar divergência entre o que é testado localmente e o
que roda no Colab.

**Procedimento manual necessário antes de rodar a célula abaixo:** o Colab não tem
acesso ao seu repositório local, então é preciso subir o pacote para o Drive.

1. No seu computador, localize a pasta `src/diabetes_sus/` do repositório
   (contém `__init__.py`, `config.py`, `idade.py`, `municipios.py`, `filtros.py`
   e os demais módulos).
2. No Google Drive, dentro de `MyDrive/diabetes_sus/`, crie a pasta `src/` e, por
   dentro dela, envie a pasta `diabetes_sus/` **inteira**, preservando a estrutura
   — ou seja, o caminho final deve ser
   `MyDrive/diabetes_sus/src/diabetes_sus/__init__.py`,
   `MyDrive/diabetes_sus/src/diabetes_sus/config.py`, e assim por diante.
   É essencial manter o `__init__.py` junto: sem ele a pasta não é reconhecida
   como um pacote Python e o `import diabetes_sus...` falha.
3. Pode ser feito arrastando a pasta pela interface web do Drive, ou sincronizando
   com o Google Drive Desktop — o importante é que a subpasta `diabetes_sus/`
   fique diretamente dentro de `{BASE}/src/`.

Depois de subir a pasta, a célula abaixo adiciona `{BASE}/src` ao `sys.path` e
importa as funções usadas na ingestão.

In [ ]:
import sys
sys.path.insert(0, f'{BASE}/src')
from diabetes_sus.filtros import filtrar_internacoes_diabetes
from diabetes_sus.idade import faixa_etaria, idade_em_anos
print('modulos carregados')

### 1.3 Lista de arquivos a baixar

O SIH/SUS publica um arquivo `.dbc` por UF/ano/mês. Com 27 UFs, 6 anos (2019-2024)
e 12 meses, são esperados 1.944 arquivos.

In [ ]:
UFS = ['AC','AL','AM','AP','BA','CE','DF','ES','GO','MA','MG','MS','MT',
       'PA','PB','PE','PI','PR','RJ','RN','RO','RR','RS','SC','SE','SP','TO']
ANOS = range(2019, 2025)

alvos = [(uf, ano, mes) for uf in UFS for ano in ANOS for mes in range(1, 13)]
print(f'{len(alvos)} arquivos esperados')   # deve imprimir 1944

### 1.4 Ingestão com checkpoint

Para cada combinação (UF, ano, mês): baixa o `.dbc` do FTP do DATASUS, descompacta
para `.dbf`, lê com `dbfread`, aplica o filtro de diabetes
(`filtrar_internacoes_diabetes`, que já exclui AIHs de continuação e marca
amputação de MMII), converte idade e faixa etária, renomeia colunas e grava um
parquet por arquivo de origem.

**A célula é idempotente:** antes de baixar, verifica se o parquet de destino já
existe e, se sim, pula o arquivo (`return 'ja_existe'`). Isso permite reexecutar a
célula quantas vezes for preciso após uma queda de sessão do Colab, sem reprocessar
o que já foi feito. Cada arquivo tem até 3 tentativas; falhas persistentes são
registradas em `logs/pendentes.json` em vez de interromper o processo inteiro —
com 1.944 downloads de um FTP público, algumas falhas transitórias são esperadas.

Este processamento é longo (pode levar horas dependendo da velocidade do FTP do
DATASUS); o Colab pode desconectar a sessão no meio — é por isso que o checkpoint
existe.

In [ ]:
import datasus_dbc, pandas as pd, urllib.request, os, json, traceback

FTP = ('https://ftp.datasus.gov.br/dissemin/publicos/SIHSUS'
       '/200801_/Dados/RD{uf}{aa:02d}{mm:02d}.dbc')

COLUNAS = ['MUNIC_RES','SEXO','IDADE','COD_IDADE','DIAG_PRINC',
           'PROC_REA','IDENT','MORTE','VAL_TOT','DIAS_PERM']

pendentes = []

def processar(uf, ano, mes):
    destino = f'{BASE}/bronze/uf={uf}/ano={ano}'
    os.makedirs(destino, exist_ok=True)
    saida = f'{destino}/RD{uf}{ano % 100:02d}{mes:02d}.parquet'
    if os.path.exists(saida):
        return 'ja_existe'

    url = FTP.format(uf=uf, aa=ano % 100, mm=mes)
    dbc, dbf = '/tmp/a.dbc', '/tmp/a.dbf'
    urllib.request.urlretrieve(url, dbc)
    datasus_dbc.decompress(dbc, dbf)

    from dbfread import DBF
    df = pd.DataFrame(iter(DBF(dbf, encoding='latin-1')))
    df = df[[c for c in COLUNAS if c in df.columns]]

    df = filtrar_internacoes_diabetes(df)
    df['idade_anos'] = idade_em_anos(df['IDADE'], df['COD_IDADE'])
    df['faixa_etaria'] = faixa_etaria(df['idade_anos']).astype(str)
    df = df.rename(columns={
        'MUNIC_RES': 'cod_municipio_6', 'SEXO': 'sexo',
        'MORTE': 'morte', 'VAL_TOT': 'val_tot', 'DIAS_PERM': 'dias_perm',
    })
    df['ano'], df['mes'] = ano, mes
    df[['cod_municipio_6','sexo','idade_anos','faixa_etaria','ano','mes',
        'amputacao','morte','val_tot','dias_perm']].to_parquet(saida, index=False)

    os.remove(dbc); os.remove(dbf)
    return f'{len(df)} linhas'

for i, (uf, ano, mes) in enumerate(alvos, 1):
    for tentativa in range(3):
        try:
            r = processar(uf, ano, mes)
            if i % 50 == 0:
                print(f'[{i}/{len(alvos)}] {uf} {ano}-{mes:02d}: {r}', flush=True)
            break
        except Exception as e:
            if tentativa == 2:
                pendentes.append({'uf': uf, 'ano': ano, 'mes': mes, 'erro': str(e)})
                print(f'FALHOU {uf} {ano}-{mes:02d}: {e}', flush=True)

with open(f'{BASE}/logs/pendentes.json', 'w') as f:
    json.dump(pendentes, f, indent=2)
print(f'concluido. pendentes: {len(pendentes)}')

### 1.5 Verificação de completude

Confere quantos dos 1.944 parquets esperados foram de fato gerados. Tolera até 2%
de arquivos faltando (ex.: meses sem AIHs em UFs pequenas, indisponibilidade
pontual do FTP); acima disso, o `assert` interrompe a execução para que
`logs/pendentes.json` seja investigado antes de seguir para a camada silver.

In [ ]:
import glob
gerados = glob.glob(f'{BASE}/bronze/uf=*/ano=*/*.parquet')
print(f'gerados: {len(gerados)} de {len(alvos)}')
assert len(gerados) >= len(alvos) * 0.98, 'completude abaixo de 98% — investigar pendentes.json'